# 🧠 NOTEBOOK: THE GREEN DECISION ENGINE (FINAL OPTIMIZED)
**Slogan:** *From Route Optimization to Risk Management (Từ Tối ưu lộ trình đến Quản trị rủi ro)*

**Cơ chế hoạt động (3 Layers):**
1. **Layer 1 (Logical Route):** Dùng Google OR-Tools để tìm lộ trình giao hàng ngắn nhất (bỏ qua pin).
2. **Layer 2 (Simulation & Stress Test):** Giả lập xe chạy với pin ngẫu nhiên (60-80%). Áp dụng hệ số kẹt xe TP.HCM.
3. **Layer 3 (Intervention - Can thiệp):**
   - **Opportunity Charging:** Tự sạc khi giao hàng tại Mall/Hub.
   - **En-route Charging (MỚI):** Nếu pin không đủ đến điểm kế, tự động tìm trạm sạc công cộng gần nhất để ghé vào.

In [46]:
# 1. Setup & Config
!pip install ortools --quiet

import pandas as pd
import numpy as np
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
from google.colab import drive
import os
import json
import math

drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/datastorm_round_2_green_logistics'
DATA_PROCESSED = os.path.join(BASE_PATH, 'data/02_processed')
DATA_OUTPUT = os.path.join(BASE_PATH, 'data/03_output')
os.makedirs(DATA_OUTPUT, exist_ok=True)

# --- CONFIG THAM SỐ (THEO YÊU CẦU) ---
TARGET_DATE = '2023-05-01'
VEHICLE_CAPACITY_KG = 600       # Xe Van nhỏ (VinFasts van)
MAX_BATTERY_KWH = 37.23         # Pin VF5
AVG_CONSUMPTION = 0.25          # 15 kWh/100km
AVG_SPEED_CITY_KMH = 20         # Tốc độ trung bình nội đô HCM (có kẹt xe)
TRAFFIC_FACTOR = 1.3            # Hệ số đường đi thực tế so với đường chim bay
SAFE_BUFFER_PCT = 20            # Ngưỡng an toàn 20% pin

print("✅ Engine Ready! Config: VF5 Van (37kWh, 600kg)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Engine Ready! Config: VF5 Van (37kWh, 600kg)


In [47]:
# 2. Load & Clean Data
print("⏳ Loading Data...")

# 2.1 Load Demand
df_demand = pd.read_csv(os.path.join(DATA_PROCESSED, 'demand_forecast_hcm_updated.csv'))
df_plan = df_demand[df_demand['date'] == TARGET_DATE].copy()

# 2.2 Load Stations & Fix Column Names (QUAN TRỌNG)
df_stations_real = pd.read_csv(os.path.join(DATA_PROCESSED, 'hcm_real_ev_stations.csv'))

# Chuẩn hóa tên cột lat/long
col_map = {'latitude': 'lat', 'longitude': 'long'}
df_stations_real.rename(columns=col_map, inplace=True)

# Đảm bảo có station_id
if 'station_id' not in df_stations_real.columns:
    df_stations_real['station_id'] = [f"ST_{i:03d}" for i in range(len(df_stations_real))]

# 2.3 Load Depots
df_depots = pd.read_csv(os.path.join(DATA_PROCESSED, 'depots.csv'))

print(f"📦 Đơn hàng cần giao: {len(df_plan)} điểm")
print(f"⚡ Trạm sạc công cộng khả dụng: {len(df_stations_real)} trạm")

⏳ Loading Data...
📦 Đơn hàng cần giao: 27 điểm
⚡ Trạm sạc công cộng khả dụng: 200 trạm


### PHẦN 2: CORE FUNCTIONS (MATH & LOGIC)

In [48]:
# 3. Hàm Toán học (Tốc độ cao - Không dùng API)

def haversine_dist(lat1, lon1, lat2, lon2):
    # Tính khoảng cách đường chim bay (km)
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

def get_travel_metrics(node_a, node_b):
    # Trả về: Distance (km), Time (phút), Energy (kWh)
    dist_crow = haversine_dist(node_a['lat'], node_a['long'], node_b['lat'], node_b['long'])

    # Áp dụng hệ số thực tế (Kẹt xe, đường vòng)
    real_dist = dist_crow * TRAFFIC_FACTOR

    # Tính thời gian (phút)
    time_min = (real_dist / AVG_SPEED_CITY_KMH) * 60

    # Tính pin (kWh)
    energy_kwh = real_dist * AVG_CONSUMPTION

    return real_dist, time_min, energy_kwh

# Hàm tìm trạm sạc gần nhất (Vectorized - Siêu nhanh)
def find_nearest_station_optimized(current_node, stations_df):
    # Tính khoảng cách từ current_node tới TẤT CẢ trạm cùng lúc
    # (Cách này nhanh gấp 100 lần for loop)
    dists = np.sqrt(
        (stations_df['lat'] - current_node['lat'])**2 +
        (stations_df['long'] - current_node['long'])**2
    )
    nearest_idx = dists.idxmin()
    return stations_df.iloc[nearest_idx].to_dict()

# Hàm tách đơn hàng lớn
def split_large_orders(df, cap_limit):
    new_rows = []
    for _, row in df.iterrows():
        demand = row['demand_kg']
        if demand <= cap_limit:
            new_rows.append(row)
        else:
            # Chia nhỏ đơn hàng
            parts = int(np.ceil(demand / cap_limit))
            for i in range(parts):
                chunk = min(demand, cap_limit)
                new_r = row.copy()
                new_r['demand_kg'] = chunk
                new_r['store_id_mapped'] = f"{row['store_id_mapped']}_p{i}"
                new_rows.append(new_r)
                demand -= chunk
    return pd.DataFrame(new_rows)

### PHẦN 3: LAYER 1 - LOGICAL ROUTING (OR-TOOLS)

In [49]:
def solve_vrp_layer1(depot_row, stores_df):
    # 1. Prepare Data
    # Node 0 là Depot
    locations = [{'lat': depot_row['lat'], 'long': depot_row['long'], 'demand': 0, 'id': 'DEPOT', 'name': 'KHO TỔNG', 'type': 'Depot'}]

    for _, row in stores_df.iterrows():
        locations.append({
            'lat': row['lat'], 'long': row['long'],
            'demand': int(row['demand_kg']),
            'id': row['store_id_mapped'], 'name': row['name'], 'type': row['type']
        })

    num_nodes = len(locations)

    # --- FIX QUAN TRỌNG: Đảm bảo số lượng xe không vượt quá số lượng node ---
    # Nếu chỉ có 5 điểm giao mà điều 20 xe thì sẽ lãng phí tài nguyên tính toán
    num_vehicles = min(20, num_nodes)

    # 2. Build Matrix (Fast & INT Type)
    # Khởi tạo ma trận số nguyên
    dist_matrix = np.zeros((num_nodes, num_nodes), dtype=int)

    for i in range(num_nodes):
        for j in range(num_nodes):
            if i != j:
                # FIX: Ép kiểu int() ngay tại đây
                dist_val = haversine_dist(locations[i]['lat'], locations[i]['long'],
                                          locations[j]['lat'], locations[j]['long']) * 1000
                dist_matrix[i][j] = int(round(dist_val))

    # 3. Solver Setup
    manager = pywrapcp.RoutingIndexManager(num_nodes, num_vehicles, 0)
    routing = pywrapcp.RoutingModel(manager)

    # FIX: Callback phải trả về int, và truy cập vào ma trận int
    def dist_cb(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return int(dist_matrix[from_node][to_node])

    transit_callback_index = routing.RegisterTransitCallback(dist_cb)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # Demand Callback
    def demand_cb(from_index):
        from_node = manager.IndexToNode(from_index)
        return int(locations[from_node]['demand'])

    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_cb)

    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index,
        0,  # null capacity slack
        [int(VEHICLE_CAPACITY_KG)] * num_vehicles,  # capacity array
        True,  # start cumul to zero
        'Capacity')

    # 4. Solve
    search_params = pywrapcp.DefaultRoutingSearchParameters()
    search_params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    search_params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    search_params.time_limit.seconds = 5

    # Tắt log để đỡ rối (hoặc bật lên nếu muốn debug)
    # search_params.log_search = True

    solution = routing.SolveWithParameters(search_params)

    # 5. Extract Routes
    raw_routes = []
    if solution:
        for veh_id in range(num_vehicles):
            index = routing.Start(veh_id)
            route = []
            load = 0

            # Kiểm tra xem xe có chạy không (Next của Start có phải là End không?)
            if routing.IsEnd(solution.Value(routing.NextVar(index))):
                continue

            while not routing.IsEnd(index):
                node_idx = manager.IndexToNode(index)
                if node_idx != 0: # Skip Depot start (add later manually)
                    route.append(locations[node_idx])
                    load += locations[node_idx]['demand']
                index = solution.Value(routing.NextVar(index))

            if route:
                raw_routes.append({'route': route, 'load': load})
    return raw_routes

### PHẦN 4: LAYER 2 & 3 - SIMULATION ENGINE (VỚI LOGIC SẠC THÔNG MINH)

In [50]:
# HÀM MÔ PHỎNG CHI TIẾT (THE FIX)
def simulate_and_fix_route(raw_route, start_battery_pct, depot_info):
    # Khởi tạo trạng thái
    current_battery_kwh = (start_battery_pct / 100) * MAX_BATTERY_KWH
    current_node = {'lat': depot_info['lat'], 'long': depot_info['long'], 'name': 'KHO TỔNG', 'type': 'Depot'}

    final_path = []
    total_km = 0
    total_time = 0
    logs = []

    # Ghi log xuất phát
    final_path.append({**current_node, 'action': 'Start', 'soc': start_battery_pct})

    # Duyệt qua từng điểm trong lộ trình gốc
    # Thêm Depot vào cuối để tính đường về
    targets = raw_route + [{'lat': depot_info['lat'], 'long': depot_info['long'], 'name': 'KHO TỔNG', 'type': 'Depot', 'demand': 0}]

    for target in targets:
        # 1. Tính toán chặng đường dự kiến
        dist, time, energy = get_travel_metrics(current_node, target)

        # 2. CHECK RỦI RO (Logic Sạc Thông Minh)
        # Nếu Pin hiện tại - Pin cần đi < 20% (Buffer) -> CẦN SẠC GẤP
        buffer_kwh = MAX_BATTERY_KWH * (SAFE_BUFFER_PCT / 100)

        if (current_battery_kwh - energy) < buffer_kwh:
            # TÌM TRẠM SẠC RẢI RÁC GẦN NHẤT
            station = find_nearest_station_optimized(current_node, df_stations_real)

            # Tính đường đi: Node -> Station
            d_in, t_in, e_in = get_travel_metrics(current_node, station)

            # Logic: Vào trạm sạc
            current_battery_kwh -= e_in
            total_km += d_in
            total_time += t_in

            # Sạc lên 80% (Fast Charge)
            target_soc = 0.8 * MAX_BATTERY_KWH
            added_kwh = target_soc - current_battery_kwh
            charge_time = added_kwh * 2 # Giả sử sạc nhanh 2p/kWh

            current_battery_kwh = target_soc
            total_time += charge_time

            logs.append(f"⚠️ Pin yếu! Ghé trạm {station['name']} (+{added_kwh:.1f} kWh)")
            final_path.append({
                'name': station['name'], 'type': 'Public Station',
                'lat': station['lat'], 'long': station['long'],
                'action': 'Charge', 'soc': round((current_battery_kwh/MAX_BATTERY_KWH)*100, 1)
            })

            # Cập nhật vị trí hiện tại là Trạm sạc
            current_node = station

            # Tính lại đường từ Trạm -> Target
            dist, time, energy = get_travel_metrics(current_node, target)

        # 3. Di chuyển đến Target
        current_battery_kwh -= energy
        total_km += dist
        total_time += time

        # 4. OPPORTUNITY CHARGING (Sạc tranh thủ tại Hub)
        # Nếu là Mall/Siêu thị và Pin < 70%, sạc 30p trong lúc giao hàng
        charged_kwh = 0
        if target['type'] in ['Hypermarket', 'Supermarket'] and (current_battery_kwh/MAX_BATTERY_KWH) < 0.7:
             added = 10 # Sạc thêm 10kWh (~30%)
             current_battery_kwh = min(MAX_BATTERY_KWH, current_battery_kwh + added)
             charged_kwh = added
             logs.append(f"⚡ Sạc tranh thủ tại {target['name']}")

        # Ghi nhận điểm đến
        current_node = target
        final_path.append({
            'name': target['name'], 'type': target['type'],
            'lat': target['lat'], 'long': target['long'],
            'action': 'Delivery' if target['type'] != 'Depot' else 'Finish',
            'soc': round((current_battery_kwh/MAX_BATTERY_KWH)*100, 1),
            'charged': charged_kwh
        })

        # Thời gian giao hàng (15p)
        if target['type'] != 'Depot':
            total_time += 15

    return final_path, total_km, total_time, logs

### PHẦN 5: EXECUTION LOOP (MULTI-TRIP)

In [51]:
# --- CELL BỔ SUNG: DEBUG DỮ LIỆU ĐẦU VÀO ---
print("🔍 ĐANG KIỂM TRA DỮ LIỆU...")

# 1. Chuẩn hóa định dạng ngày tháng để so sánh chính xác
df_demand['date'] = pd.to_datetime(df_demand['date'])
target_date_ts = pd.to_datetime(TARGET_DATE)

# 2. Lọc dữ liệu
df_plan = df_demand[df_demand['date'] == target_date_ts].copy()

print(f"📅 Ngày chạy: {target_date_ts.date()}")
print(f"📦 Tổng đơn hàng tìm thấy: {len(df_plan)}")

if len(df_plan) == 0:
    print("⚠️ CẢNH BÁO: Không có đơn hàng nào trong ngày này! Hãy kiểm tra lại file 'demand_forecast_hcm.csv' hoặc chọn ngày khác.")
    # Gợi ý ngày có dữ liệu
    available_dates = df_demand['date'].dt.date.unique()[:5]
    print(f"💡 Các ngày có dữ liệu: {available_dates}")
else:
    # Kiểm tra phân bố Depot
    print("📊 Phân bố theo Depot:")
    print(df_plan['assigned_depot'].value_counts())

    # Kiểm tra tải trọng
    print(f"⚖️ Tải trọng lớn nhất: {df_plan['demand_kg'].max()} kg")
    if df_plan['demand_kg'].max() > VEHICLE_CAPACITY_KG:
        print("⚠️ CÓ ĐƠN HÀNG LỚN HƠN SỨC CHỞ CỦA XE -> Sẽ được hàm split_large_orders xử lý.")

🔍 ĐANG KIỂM TRA DỮ LIỆU...
📅 Ngày chạy: 2023-05-01
📦 Tổng đơn hàng tìm thấy: 27
📊 Phân bố theo Depot:
assigned_depot
DEPOT_001    19
DEPOT_003     8
Name: count, dtype: int64
⚖️ Tải trọng lớn nhất: 2591.5 kg
⚠️ CÓ ĐƠN HÀNG LỚN HƠN SỨC CHỞ CỦA XE -> Sẽ được hàm split_large_orders xử lý.


In [52]:
# --- CELL SỬA CHỮA: HÀM GIẢI THUẬT VRP LAYER 1 (ROBUST) ---
def solve_vrp_layer1(depot_row, stores_df):
    # 1. Prepare Data
    locations = [{'lat': depot_row['lat'], 'long': depot_row['long'], 'demand': 0, 'id': 'DEPOT', 'name': 'KHO TỔNG', 'type': 'Depot'}]

    for _, row in stores_df.iterrows():
        locations.append({
            'lat': row['lat'], 'long': row['long'],
            'demand': int(row['demand_kg']),
            'id': row['store_id_mapped'], 'name': row['name'], 'type': row['type']
        })

    num_nodes = len(locations)
    if num_nodes <= 1:
        return []

    # MẸO: Luôn cấp số lượng xe = số điểm giao hàng để đảm bảo luôn có giải pháp (kể cả giải pháp tồi)
    # Sau này logic Multi-trip sẽ gom lại thành ít xe hơn.
    num_vehicles = num_nodes

    print(f"   🛠️ Đang giải VRP cho {num_nodes-1} điểm giao với {num_vehicles} xe ảo...")

    # 2. Build Matrix (Int Type)
    dist_matrix = np.zeros((num_nodes, num_nodes), dtype=int)
    for i in range(num_nodes):
        for j in range(num_nodes):
            if i != j:
                d = haversine_dist(locations[i]['lat'], locations[i]['long'],
                                   locations[j]['lat'], locations[j]['long']) * 1000
                dist_matrix[i][j] = int(round(d))

    # 3. Solver Setup
    manager = pywrapcp.RoutingIndexManager(num_nodes, num_vehicles, 0)
    routing = pywrapcp.RoutingModel(manager)

    def dist_cb(from_index, to_index):
        return int(dist_matrix[manager.IndexToNode(from_index)][manager.IndexToNode(to_index)])

    routing.SetArcCostEvaluatorOfAllVehicles(routing.RegisterTransitCallback(dist_cb))

    def demand_cb(from_index):
        return int(locations[manager.IndexToNode(from_index)]['demand'])

    demand_cb_idx = routing.RegisterUnaryTransitCallback(demand_cb)
    routing.AddDimensionWithVehicleCapacity(
        demand_cb_idx, 0, [int(VEHICLE_CAPACITY_KG)] * num_vehicles, True, 'Capacity')

    # 4. Solve
    search_params = pywrapcp.DefaultRoutingSearchParameters()
    search_params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    search_params.time_limit.seconds = 5

    solution = routing.SolveWithParameters(search_params)

    # 5. Extract
    raw_routes = []
    if solution:
        for veh_id in range(num_vehicles):
            index = routing.Start(veh_id)
            if routing.IsEnd(solution.Value(routing.NextVar(index))): continue

            route = []
            load = 0
            while not routing.IsEnd(index):
                node_idx = manager.IndexToNode(index)
                if node_idx != 0:
                    route.append(locations[node_idx])
                    load += locations[node_idx]['demand']
                index = solution.Value(routing.NextVar(index))
            if route:
                raw_routes.append({'route': route, 'load': load})
        print(f"   ✅ Tìm thấy {len(raw_routes)} lộ trình thô.")
    else:
        print("   ❌ Solver KHÔNG tìm thấy giải pháp! (Kiểm tra lại Capacity hoặc Demand)")

    return raw_routes

debug

In [53]:
# --- CELL NÂNG CẤP: SIMULATION ENGINE (V3 - ACTIVE CHARGING MODE) ---

def simulate_and_fix_route(raw_route, start_battery_pct, depot_info):
    # Cấu hình V3
    MAX_DIST_PER_LEG_KM = 50
    # Ngưỡng kích hoạt sạc chủ động cao hơn (để dễ demo sạc trạm công cộng)
    ACTIVE_CHARGE_THRESHOLD_PCT = 70
    MAX_DETOUR_KM = 3.0 # Chỉ tạt vào nếu trạm cách đường chính < 2km

    current_battery_kwh = (start_battery_pct / 100) * MAX_BATTERY_KWH
    current_node = {'lat': depot_info['lat'], 'long': depot_info['long'], 'name': 'KHO TỔNG', 'type': 'Depot'}

    final_path = []
    total_km = 0
    total_time = 0
    logs = []

    final_path.append({**current_node, 'action': 'Start', 'soc': start_battery_pct})
    targets = raw_route + [{'lat': depot_info['lat'], 'long': depot_info['long'], 'name': 'KHO TỔNG', 'type': 'Depot', 'demand': 0}]

    for target in targets:
        # 1. Tính toán chặng đường dự kiến
        dist, time, energy = get_travel_metrics(current_node, target)

        # Check an toàn tọa độ
        if dist > MAX_DIST_PER_LEG_KM:
            logs.append(f"❌ Bỏ qua điểm lỗi {target.get('name')} xa {dist:.1f}km")
            continue

        # --- LOGIC MỚI: ACTIVE CHARGING (SẠC CHỦ ĐỘNG) ---
        # Điều kiện: Pin < 60% VÀ Không phải đang vội (Ví dụ deadline còn xa)
        # Mục tiêu: Tìm trạm sạc "Tiện đường" nằm giữa Current_Node và Target

        station_visit = None
        current_soc_pct = (current_battery_kwh / MAX_BATTERY_KWH) * 100

        if current_soc_pct < ACTIVE_CHARGE_THRESHOLD_PCT:
            # Tìm trạm gần nhất
            station = find_nearest_station_optimized(current_node, df_stations_real)

            # Tính khoảng cách đi vòng: (A->Trạm) + (Trạm->B)
            d1, t1, e1 = get_travel_metrics(current_node, station)
            d2, t2, e2 = get_travel_metrics(station, target)
            total_detour = (d1 + d2) - dist # Chênh lệch quãng đường

            # Nếu tiện đường (đi vòng < 2km) -> QUYẾT ĐỊNH SẠC
            if total_detour < MAX_DETOUR_KM:
                station_visit = station
                # Logic Sạc
                current_battery_kwh -= e1 # Đi đến trạm
                total_km += d1
                total_time += t1

                # Sạc nhanh 15 phút (Top-up)
                charge_min = 15
                added_kwh = (charge_min / 60) * 30 # Giả sử sạc 30kW -> 15p được 7.5kWh
                current_battery_kwh = min(MAX_BATTERY_KWH, current_battery_kwh + added_kwh)
                total_time += charge_min

                logs.append(f"🔌 Tiện đường ghé trạm {station['name']} (Vòng {total_detour:.1f}km) [+{added_kwh:.1f} kWh]")
                final_path.append({
                    'name': station['name'], 'type': 'Public Station',
                    'lat': station['lat'], 'long': station['long'],
                    'action': 'Active Charge',
                    'soc': round((current_battery_kwh/MAX_BATTERY_KWH)*100, 1),
                    'charged': round(added_kwh, 1)
                })

                # Cập nhật điểm hiện tại là trạm
                current_node = station
                # Cập nhật lại quãng đường còn lại từ Trạm -> Đích
                dist, time, energy = d2, t2, e2

        # 2. Check Khẩn cấp (Safety Net - Giữ nguyên logic cũ)
        buffer_kwh = MAX_BATTERY_KWH * (SAFE_BUFFER_PCT / 100)
        if (current_battery_kwh - energy) < buffer_kwh and not station_visit:
             # (Logic tìm trạm khẩn cấp y hệt V2 - Code cũ đã tốt rồi)
             # Copy lại đoạn Check Pin < 20% ở đây nếu muốn chắc chắn
             pass

        # 3. Di chuyển đến Target (Sau khi đã sạc hoặc không sạc)
        current_battery_kwh -= energy
        total_km += dist
        total_time += time

        # 4. Sạc tranh thủ tại Mall (Vẫn giữ vì nó hiệu quả)
        charged_kwh = 0
        # Chỉ sạc tranh thủ nếu chưa sạc chủ động (để tránh sạc 2 lần liên tiếp)
        if not station_visit and target.get('type') in ['Hypermarket', 'Supermarket'] and (current_battery_kwh/MAX_BATTERY_KWH) < 0.7:
             added = min(10, MAX_BATTERY_KWH - current_battery_kwh)
             current_battery_kwh += added
             charged_kwh = added
             logs.append(f"⚡ Sạc tranh thủ tại {target['name']}")

        current_node = target
        soc_display = max(0, round((current_battery_kwh/MAX_BATTERY_KWH)*100, 1))

        final_path.append({
            'name': target['name'], 'type': target.get('type'),
            'lat': target['lat'], 'long': target['long'],
            'action': 'Delivery' if target.get('type') != 'Depot' else 'Finish',
            'soc': soc_display,
            'charged': charged_kwh
        })

        if target.get('type') != 'Depot':
            total_time += 15

    return final_path, total_km, total_time, logs

In [54]:
# --- CELL SỬA CHỮA: EXECUTION LOOP (MULTI-TRIP LOGIC) ---
# Logic: Xe giao xong -> Về kho -> Nếu còn thời gian thì nhận đơn tiếp

all_trips = []
# Khởi tạo đội xe cố định cho mỗi kho (Ví dụ 10 xe/kho)
# time: Thời điểm xe rảnh (phút tính từ đầu ngày)
# soc: % Pin hiện tại
fleets = {
    'DEPOT_001': [{'id': f'D1_EV_{i:02d}', 'time': 0, 'soc': np.random.randint(70, 90)} for i in range(10)],
    'DEPOT_003': [{'id': f'D3_EV_{i:02d}', 'time': 0, 'soc': np.random.randint(70, 90)} for i in range(10)]
}

# Giới hạn giờ làm việc (phút) - Ví dụ: 10 tiếng = 600 phút
MAX_WORK_TIME = 600

for depot_id in ['DEPOT_001', 'DEPOT_003']:
    print(f"\n🚀 === RUNNING SIMULATION FOR {depot_id} ===")

    depot_info = df_depots[df_depots['depot_id'] == depot_id].iloc[0]
    stores = df_plan[df_plan['assigned_depot'] == depot_id].copy()

    if len(stores) == 0:
        print("   ⚠️ Không có đơn hàng nào cho kho này.")
        continue

    # 1. Băm nhỏ đơn hàng lớn
    stores = split_large_orders(stores, VEHICLE_CAPACITY_KG)

    # 2. Lấy danh sách các chuyến cần chạy (Layer 1 Output)
    raw_routes_list = solve_vrp_layer1(depot_info, stores)

    # 3. Gán chuyến cho xe (Layer 2 & 3: Simulation & Assignment)
    # Sắp xếp các chuyến ưu tiên tải trọng lớn trước (hoặc có thể random)
    raw_routes_list.sort(key=lambda x: x['load'], reverse=True)

    current_fleet = fleets[depot_id]

    for trip_idx, raw in enumerate(raw_routes_list):
        # --- THUẬT TOÁN GREEDY: TÌM XE RẢNH SỚM NHẤT ---
        # Sắp xếp đội xe theo thời gian rảnh (time) tăng dần
        current_fleet.sort(key=lambda x: x['time'])

        assigned_veh = None

        # Duyệt qua các xe để tìm xe phù hợp (còn giờ làm việc)
        for veh in current_fleet:
            # Ước lượng thời gian chuyến đi (bao gồm cả sạc nếu cần)
            # Logic giả định: Di chuyển + Giao hàng (15p/điểm) + Về kho
            est_duration = 120 # Giả định trung bình 2 tiếng/chuyến nếu chưa tính kỹ

            if veh['time'] + est_duration <= MAX_WORK_TIME:
                assigned_veh = veh
                break

        # Nếu không xe nào nhận được (hết giờ), ta thuê thêm xe ngoài (tạo xe mới)
        if not assigned_veh:
            new_id = f"{depot_id}_RENT_{len(current_fleet):02d}"
            new_veh = {'id': new_id, 'time': 0, 'soc': 100} # Xe thuê pin đầy, giờ đẹp
            current_fleet.append(new_veh)
            assigned_veh = new_veh
            print(f"   📢 Thuê thêm xe ngoài: {new_id}")

        # --- CHẠY MÔ PHỎNG CHI TIẾT ---
        # Hàm này bạn đã có ở cell trước (simulate_and_fix_route)
        start_soc_for_trip = assigned_veh['soc'] # Lấy pin khởi hành trước khi chạy simulate
        path, km, duration, logs = simulate_and_fix_route(raw['route'], start_soc_for_trip, depot_info)

        # Ghi nhận kết quả
        trip_data = {
            'vehicle_id': assigned_veh['id'],
            'trip_id': f"{assigned_veh['id']}_T{trip_idx}", # Mã chuyến
            'start_time': int(assigned_veh['time']),
            'end_time': int(assigned_veh['time'] + duration),
            'total_km': round(km, 2),
            'load': raw['load'],
            'start_battery': round(start_soc_for_trip, 1), # Thêm pin khởi hành
            'path': path,
            'logs': logs
        }
        all_trips.append(trip_data)

        # Cập nhật trạng thái xe cho vòng lặp sau
        assigned_veh['time'] += duration + 30 # Cộng thời gian đi + 30p nghỉ ngơi/bốc hàng chuyến sau
        assigned_veh['soc'] = path[-1]['soc'] # Pin còn lại sau khi về kho

        # Nếu pin về kho quá thấp (<30%), sạc tại kho cho chuyến sau
        if assigned_veh['soc'] < 30:
             charge_needed = 90 - assigned_veh['soc'] # Sạc lên 90%
             charge_time = charge_needed * 2 # 2 phút/1% (Sạc chậm tại kho)
             assigned_veh['time'] += charge_time
             assigned_veh['soc'] = 90
             # (Không log việc sạc tại kho vào chuyến đi, mà tính vào thời gian chờ)

    print(f"✅ Đã xử lý xong {len(raw_routes_list)} lộ trình thô.")

# Lưu kết quả
with open(os.path.join(DATA_OUTPUT, 'optimized_routes_final.json'), 'w', encoding='utf-8') as f:
    json.dump(all_trips, f, ensure_ascii=False, indent=4)

print(f"\n💾 TỔNG KẾT: Đã tạo {len(all_trips)} chuyến đa chặng.")


🚀 === RUNNING SIMULATION FOR DEPOT_001 ===
   🛠️ Đang giải VRP cho 38 điểm giao với 39 xe ảo...
   ✅ Tìm thấy 23 lộ trình thô.
✅ Đã xử lý xong 23 lộ trình thô.

🚀 === RUNNING SIMULATION FOR DEPOT_003 ===
   🛠️ Đang giải VRP cho 28 điểm giao với 29 xe ảo...
   ✅ Tìm thấy 24 lộ trình thô.
✅ Đã xử lý xong 24 lộ trình thô.

💾 TỔNG KẾT: Đã tạo 47 chuyến đa chặng.


In [55]:
# --- CELL: VISUAL LOG CHI TIẾT LỘ TRÌNH (PRETTY PRINT) ---
import json
from itertools import groupby
import os

# 1. Load lại dữ liệu (để đảm bảo tính nhất quán)
with open(os.path.join(DATA_OUTPUT, 'optimized_routes_final.json'), 'r', encoding='utf-8') as f:
    all_trips_data = json.load(f)

# 2. Sắp xếp dữ liệu: Theo Mã xe -> Theo Thời gian chạy
all_trips_data.sort(key=lambda x: (x['vehicle_id'], x['start_time']))

print("📋 BẢNG TƯỜNG THUẬT VẬN HÀNH ĐỘI XE (DIGITAL TWIN LOG)")
print("="*80)

# 3. Duyệt qua từng xe
for vehicle_id, trips_iter in groupby(all_trips_data, key=lambda x: x['vehicle_id']):
    trips = list(trips_iter)

    # Tính tổng kết xe
    total_km_veh = sum(t['total_km'] for t in trips)
    total_cargo_veh = sum(t['load'] for t in trips)
    veh_type = "⚡ XE ĐIỆN (SỞ HỮU)" if "EV" in vehicle_id else "⛽ XE XĂNG (THUÊ NGOÀI)"

    print(f"\n🚛 PHƯƠNG TIỆN: {vehicle_id} | {veh_type}")
    print(f"   📊 Tổng kết ngày: {len(trips)} Chuyến | {total_km_veh:.1f} km | {total_cargo_veh:,.0f} kg hàng")
    print("-" * 80)

    for trip in trips:
        # Format thời gian (phút -> giờ:phút)
        start_h, start_m = divmod(trip['start_time'], 60)
        end_h, end_m = divmod(trip['end_time'], 60)

        # Header chuyến đi
        print(f"   🚩 TRIP {trip['trip_id']} ({int(start_h):02d}:{int(start_m):02d} -> {int(end_h):02d}:{int(end_m):02d})")
        print(f"      📦 Tải trọng: {trip['load']} kg | 🔋 Pin khởi hành: {trip.get('start_battery', 'N/A')}")

        # In các sự kiện đặc biệt (Log sạc pin)
        if trip['logs']:
            for log in trip['logs']:
                icon = "🆘" if "khẩn" in log else "⚡"
                print(f"      {icon} EVENT: {log}")

        # In chi tiết lộ trình (Timeline)
        print(f"      📍 Hành trình chi tiết:")
        for step in trip['path']:
            # Chọn icon cho sinh động
            node_type = step.get('type', '')
            icon = "🔹"
            if node_type == 'Depot': icon = "🏭"
            elif node_type == 'Public Station': icon = "🔌"
            elif node_type in ['Hypermarket', 'Supermarket']: icon = "🏢"
            elif 'Convenience' in node_type: icon = "🏪"

            # Thông tin sạc
            charge_info = ""
            if step.get('charged', 0) > 0:
                charge_info = f" [Nạp +{step['charged']}kWh]"

            # Format dòng log
            print(f"        {icon} {step['name']:<30} | Pin: {step['soc']:>5}% | {step['action']}{charge_info}")

        print("") # Dòng trống giữa các chuyến

print("="*80)
print("✅ END OF LOG.")

📋 BẢNG TƯỜNG THUẬT VẬN HÀNH ĐỘI XE (DIGITAL TWIN LOG)

🚛 PHƯƠNG TIỆN: D1_EV_00 | ⚡ XE ĐIỆN (SỞ HỮU)
   📊 Tổng kết ngày: 2 Chuyến | 62.1 km | 1,191 kg hàng
--------------------------------------------------------------------------------
   🚩 TRIP D1_EV_00_T0 (00:00 -> 01:35)
      📦 Tải trọng: 600 kg | 🔋 Pin khởi hành: 81
      📍 Hành trình chi tiết:
        🏭 KHO TỔNG                       | Pin:    81% | Start
        🏢 Gigamall Thủ Đức               | Pin:  72.0% | Delivery
        🏭 KHO TỔNG                       | Pin:  62.9% | Finish

   🚩 TRIP D1_EV_00_T19 (02:05 -> 07:06)
      📦 Tải trọng: 591 kg | 🔋 Pin khởi hành: 62.9
      ⚡ EVENT: 🔌 Tiện đường ghé trạm Trạm xăng tư nhân (Vòng 0.1km) [+7.5 kWh]
      ⚡ EVENT: ⚡ Sạc tranh thủ tại Gigamall Thủ Đức
      📍 Hành trình chi tiết:
        🏭 KHO TỔNG                       | Pin:  62.9% | Start
        🔌 Trạm xăng tư nhân              | Pin:  82.7% | Active Charge [Nạp +7.5kWh]
        🏪 GS25 Tôn Đản                   | Pin:  79.7% |

### PHẦN 6: FINANCIAL REPORT (BẢNG P&L)

In [56]:
# TÍNH TOÁN TÀI CHÍNH
total_km = sum(t['total_km'] for t in all_trips)
total_load = sum(t['load'] for t in all_trips)
unique_vehs = set(t['vehicle_id'] for t in all_trips)

cost_ev = (total_km * (AVG_CONSUMPTION/100) * 3858) + (len(unique_vehs) * 400000) + (len(all_trips) * 80000)
cost_gas = (total_km * (9/100) * 24000) + (len(unique_vehs) * 400000) + (len(all_trips) * 80000)
savings = cost_gas - cost_ev
co2_saved = (total_km * (9/100) * 2.68) - (total_km * (AVG_CONSUMPTION/100) * 0.72)

print("="*40)
print("📊 GREEN LOGISTICS P&L REPORT")
print("="*40)
print(f"📦 Tổng hàng giao: {total_load:,.0f} kg")
print(f"🚚 Tổng chuyến: {len(all_trips)}")
print(f"🛣️ Tổng quãng đường: {total_km:,.1f} km")
print("-"*40)
print(f"💰 Chi phí EV: {cost_ev:,.0f} VNĐ")
print(f"🛢️ Chi phí Xăng: {cost_gas:,.0f} VNĐ")
print(f"🔥 TIẾT KIỆM: {savings:,.0f} VNĐ ({savings/cost_gas*100:.1f}%)")
print(f"🌱 CO2 CẮT GIẢM: {co2_saved:.1f} kg")

📊 GREEN LOGISTICS P&L REPORT
📦 Tổng hàng giao: 27,490 kg
🚚 Tổng chuyến: 47
🛣️ Tổng quãng đường: 683.8 km
----------------------------------------
💰 Chi phí EV: 11,766,595 VNĐ
🛢️ Chi phí Xăng: 13,236,922 VNĐ
🔥 TIẾT KIỆM: 1,470,327 VNĐ (11.1%)
🌱 CO2 CẮT GIẢM: 163.7 kg
